In [1]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import os
from datetime import datetime

class EquipmentReconApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Equipment Reconciliation Tool")
        self.root.geometry("600x500")
        
        # Styles for a cleaner look
        style = ttk.Style()
        style.configure("TButton", padding=6, relief="flat", background="#ccc")
        style.configure("TLabel", padding=5, font=("Helvetica", 10))

        # Variables to store file paths
        self.source_path = tk.StringVar()
        self.master_path = tk.StringVar()
        self.export_folder = tk.StringVar()

        # --- GUI LAYOUT ---
        # Title
        title_label = ttk.Label(root, text="Equipment Data Reconciliation", font=("Helvetica", 14, "bold"))
        title_label.pack(pady=15)

        # Container for form inputs
        form_frame = ttk.Frame(root, padding=20)
        form_frame.pack(fill='both', expand=True)

        # 1. Source File Selection
        ttk.Label(form_frame, text="Source File (UCSD EQUIPMENT Summary):").grid(row=0, column=0, sticky='w')
        ttk.Entry(form_frame, textvariable=self.source_path, width=50).grid(row=1, column=0, padx=5, pady=5)
        ttk.Button(form_frame, text="Browse", command=self.select_source).grid(row=1, column=1, padx=5)

        # 2. Master File Selection
        ttk.Label(form_frame, text="Master File (Part Numbers Map):").grid(row=2, column=0, sticky='w', pady=(10, 0))
        ttk.Entry(form_frame, textvariable=self.master_path, width=50).grid(row=3, column=0, padx=5, pady=5)
        ttk.Button(form_frame, text="Browse", command=self.select_master).grid(row=3, column=1, padx=5)

        # 3. Export Folder Selection
        ttk.Label(form_frame, text="Export Folder:").grid(row=4, column=0, sticky='w', pady=(10, 0))
        ttk.Entry(form_frame, textvariable=self.export_folder, width=50).grid(row=5, column=0, padx=5, pady=5)
        ttk.Button(form_frame, text="Browse", command=self.select_folder).grid(row=5, column=1, padx=5)

        # 4. Action Buttons
        btn_frame = ttk.Frame(root, padding=20)
        btn_frame.pack(fill='x')
        self.run_btn = tk.Button(btn_frame, text="RUN RECONCILIATION", bg="#4CAF50", fg="white", 
                                 font=("Helvetica", 10, "bold"), height=2, command=self.run_process)
        self.run_btn.pack(fill='x')

        # Status Bar
        self.status_var = tk.StringVar()
        self.status_var.set("Ready")
        status_bar = tk.Label(root, textvariable=self.status_var, bd=1, relief=tk.SUNKEN, anchor='w')
        status_bar.pack(side=tk.BOTTOM, fill=tk.X)

    # --- HELPER FUNCTIONS ---

    def select_source(self):
        filename = filedialog.askopenfilename(title="Select Source File", filetypes=[("Excel Files", "*.xlsx *.xls")])
        if filename: self.source_path.set(filename)

    def select_master(self):
        filename = filedialog.askopenfilename(title="Select Master File", filetypes=[("Excel Files", "*.xlsx *.xls")])
        if filename: self.master_path.set(filename)

    def select_folder(self):
        folder = filedialog.askdirectory(title="Select Export Folder")
        if folder: self.export_folder.set(folder)

    # --- CORE LOGIC ---

    def run_process(self):
        # 1. Validation
        if not all([self.source_path.get(), self.master_path.get(), self.export_folder.get()]):
            messagebox.showerror("Error", "Please select both input files and an export folder.")
            return

        try:
            self.status_var.set("Processing data...")
            self.root.update_idletasks() # Force UI update        

            # --- STEP A: Load Data ---
            # Using 'converters' to ensure Account No is read as string initially to prevent float conversion errors          
            try:
                # sheet_name=0 loads the first sheet
                # header=16 tells pandas the column names are on Row 17 (0-indexed)
                df_source = pd.read_excel(self.source_path.get(), sheet_name=0, header=16, dtype={'ACCOUNT NO.': str})
            except Exception as e:
                messagebox.showerror("Error", f"Could not read the first sheet of the Source File.\nDetails: {e}")
                self.status_var.set("Error: Read Failed")
                return
            
            df_master = pd.read_excel(self.master_path.get())

            # --- STEP B: Data Cleaning (Source) ---
            # 1. Standardize 'ACCOUNT NO.': Convert to string -> Strip whitespace -> Lstrip '0'
            # We treat NaN as empty string to avoid errors
            df_source['ACCOUNT NO.'] = df_source['ACCOUNT NO.'].fillna('').astype(str).str.strip().str.lstrip('0')

            # 2. Remove Duplicates
            # EDIT: Changed logic to drop only exact duplicates. 
            # Previous logic dropped duplicates based solely on 'ACCOUNT NO.', which deletes 
            # valid equipment if a department has multiple units.
            initial_count = len(df_source)
            # drop_duplicates() without arguments checks for identical rows across all columns
            df_source.drop_duplicates(inplace=True)
            dropped_count = initial_count - len(df_source)
            print(f"Removed {dropped_count} fully duplicate rows.")

            # --- STEP C: Matching Logic ---
            # We strip whitespace from the join keys to ensure clean matching
            df_source['EQUIP_CLEAN'] = df_source['EQUIP'].astype(str).str.strip()
            df_master['Equipment_CLEAN'] = df_master['Equipment'].astype(str).str.strip()

            # Left Merge: Keep all source rows, attach Master data where matches occur
            df_merged = pd.merge(
                df_source, 
                df_master[['Equipment_CLEAN', 'Part No.']], 
                left_on='EQUIP_CLEAN', 
                right_on='Equipment_CLEAN', 
                how='left'
            )

            # --- STEP D: Formatting Output ---
            # Rename 'QTY' (or 'Qty') to 'Quantity' as requested in output structure
            # EDIT: Added 'QTY' to the dictionary because the source file uses all-caps 'QTY'.
            rename_map = {'Qty': 'Quantity', 'QTY': 'Quantity', 'qty': 'Quantity'}
            df_merged.rename(columns=rename_map, inplace=True)
            
            # Handle cases where Part No. might be NaN (no match found)
            df_merged['Part No.'] = df_merged['Part No.'].fillna('NO MATCH FOUND')

            # Select and Reorder specific columns
            # Using .get() ensures the script doesn't crash if a column is missing
            final_columns = ['ACCOUNT NO.', 'EQUIP', 'Quantity', 'Part No.']
            
            # Check if columns exist
            missing_cols = [c for c in final_columns if c not in df_merged.columns]
            if missing_cols:
                # If columns are missing, it's likely a file format issue.
                raise ValueError(f"Missing columns in source data after processing: {missing_cols}")

            df_final = df_merged[final_columns].copy()

            # ==========================================
            # NEW LOGIC: SPLIT DATA BASED ON FREQUENCY
            # ==========================================
            
            # 1. Analyze Frequency: Calculate how many times each Account No appears
            account_counts = df_final['ACCOUNT NO.'].value_counts()
            
            # Map these counts back to the main dataframe
            df_final['Freq_Count'] = df_final['ACCOUNT NO.'].map(account_counts)

            # 2. Split the Data
            
            # DataFrame 1 (df_unique): Account Number appears exactly once
            df_unique = df_final[df_final['Freq_Count'] == 1].drop(columns=['Freq_Count'])

            # DataFrame 2 (df_duplicates): Account Number appears 2 or more times
            # We sort this by Account NO. so duplicates appear next to each other for easy reading
            df_duplicates = df_final[df_final['Freq_Count'] >= 2].sort_values(by='ACCOUNT NO.').drop(columns=['Freq_Count'])

            # --- STEP E: Export ---
            timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
            file_name = f"Equipment_Recon_File_{timestamp}.xlsx"
            full_path = os.path.join(self.export_folder.get(), file_name)

            # Using ExcelWriter to save two sheets in one file
            with pd.ExcelWriter(full_path, engine='openpyxl') as writer:
                df_unique.to_excel(writer, sheet_name='Unique Accounts', index=False)
                df_duplicates.to_excel(writer, sheet_name='Multiple Occurrences', index=False)

            self.status_var.set("Ready")
            messagebox.showinfo("Success", f"File created successfully with 2 sheets!\n\nSaved at:\n{full_path}")

        except Exception as e:
            self.status_var.set("Error")
            messagebox.showerror("Execution Error", f"An error occurred:\n{str(e)}")
            print(e)

if __name__ == "__main__":
    root = tk.Tk()
    app = EquipmentReconApp(root)
    root.mainloop()

c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Removed 4 fully duplicate rows.


KeyboardInterrupt: 